In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

print(f"PyTorch version: {torch.__version__}")
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
device

PyTorch version: 2.11.0


device(type='mps')

In [2]:
CONFIG = {
    'window_size': 24,
    'lstm_units': 64,
    'num_layers': 1,
    'dropout': 0.2,
    'epochs': 150,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'patience': 20,
}

In [3]:
df = pd.read_csv("full_dataset_featured.csv", parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
df.head(10)

,datetime,carbon_intensity,cfe_pct,re_pct,region,temp_c,wind_speed,cloud_cover,solar_radiation,hour,...,carbon_intensity_lag_1,carbon_intensity_lag_2,carbon_intensity_lag_3,carbon_intensity_lag_24,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos
0,2023-01-02 00:00:00,189.75,57.76,42.56,california,13.8,15.7,62,214.0,0,...,147.89,124.07,122.53,214.40,0.000000,1.000000,0.0,1.0,0.5,0.866025
1,2023-01-02 00:00:00,509.93,28.71,9.96,illinois,4.2,3.9,100,0.0,0,...,504.67,503.62,494.14,503.12,0.000000,1.000000,0.0,1.0,0.5,0.866025
2,2023-01-02 00:00:00,283.57,51.04,38.79,texas,19.0,15.1,100,9.0,0,...,343.38,325.52,293.79,299.71,0.000000,1.000000,0.0,1.0,0.5,0.866025
3,2023-01-02 00:00:00,301.53,44.60,5.90,virginia,9.6,10.0,45,0.0,0,...,302.36,298.29,301.83,306.14,0.000000,1.000000,0.0,1.0,0.5,0.866025
4,2023-01-02 01:00:00,509.50,28.80,9.69,illinois,3.3,9.7,100,0.0,1,...,509.93,504.67,503.62,469.22,0.258819,0.965926,0.0,1.0,0.5,0.866025
5,2023-01-02 01:00:00,225.68,61.28,48.93,texas,18.9,17.8,100,0.0,1,...,283.57,343.38,325.52,225.67,0.258819,0.965926,0.0,1.0,0.5,0.866025
6,2023-01-02 01:00:00,305.39,43.81,4.81,virginia,9.1,9.0,94,0.0,1,...,301.53,302.36,298.29,296.97,0.258819,0.965926,0.0,1.0,0.5,0.866025
7,2023-01-02 01:00:00,207.79,53.13,38.72,california,11.4,12.8,19,54.0,1,...,189.75,147.89,124.07,216.88,0.258819,0.965926,0.0,1.0,0.5,0.866025
8,2023-01-02 02:00:00,213.71,52.04,37.98,california,9.6,10.7,57,0.0,2,...,207.79,189.75,147.89,220.01,0.500000,0.866025,0.0,1.0,0.5,0.866025
9,2023-01-02 02:00:00,304.45,44.53,4.32,virginia,8.4,7.7,93,0.0,2,...,305.39,301.53,302.36,291.99,0.500000,0.866025,0.0,1.0,0.5,0.866025


In [4]:
feature_cols = [
    # weather
    "temp_c", "wind_speed", "cloud_cover", "solar_radiation",
    # lag features
    "carbon_intensity_lag_1", "carbon_intensity_lag_2",
    "carbon_intensity_lag_3", "carbon_intensity_lag_24",
    # cyclical time encodings
    "hour_sin", "hour_cos",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos",
    # energy mix
    "cfe_pct", "re_pct"
]

target_col = 'carbon_intensity'

In [5]:
def create_sequences(X, y, window_size):
    Xs, ys = [], []
    for i in range(len(X) - window_size):
        Xs.append(X[i : i + window_size])    # window_size, n_features
        ys.append(y[i + window_size])        # next step target
    return np.array(Xs), np.array(ys)

def make_splits(group, feature_cols, target_col, window_size, feature_scaler, target_scaler):
    group = group.sort_values('datetime').reset_index(drop=True)

    X = group[feature_cols].values
    y = group[target_col].values.reshape(-1, 1)

    n = len(X)
    train_end = int(n * 0.8)

    X_train_raw, y_train_raw = X[:train_end], y[:train_end]
    X_val_raw, y_val_raw   = X[train_end:], y[train_end:]


    X_train = feature_scaler.fit_transform(X_train_raw)
    X_val = feature_scaler.transform(X_val_raw)

    y_train = target_scaler.fit_transform(y_train_raw).flatten().astype(np.float32)
    y_val = target_scaler.transform(y_val_raw).flatten().astype(np.float32)

    X_train_seq, y_train_seq = create_sequences(X_train, y_train, window_size)
    X_val_seq, y_val_seq = create_sequences(X_val, y_val, window_size)

    return X_train_seq, y_train_seq, X_val_seq, y_val_seq, feature_scaler, target_scaler


def make_virginia_test(group, feature_cols, target_col, window_size, feature_scaler, target_scaler):
    group = group.sort_values('datetime').reset_index(drop=True)

    X = group[feature_cols].values
    y = group[target_col].values.reshape(-1, 1)

    X_test = feature_scaler.transform(X)
    y_test = target_scaler.transform(y).flatten().astype(np.float32)

    X_test_seq, y_test_seq = create_sequences(X_test, y_test, window_size)
    return X_test_seq, y_test_seq

In [6]:
class CarbonDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
class LSTM(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out).squeeze(-1)

In [8]:
def train_model(X_train_seq, y_train_seq, X_val_seq, y_val_seq, config, device, save_path):

    train_loader = DataLoader(CarbonDataset(X_train_seq, y_train_seq),
                              batch_size=config['batch_size'], shuffle=False)
    val_loader = DataLoader(CarbonDataset(X_val_seq, y_val_seq),
                              batch_size=config['batch_size'], shuffle=False)

    model = LSTM(X_train_seq.shape[2], config['lstm_units'], config['num_layers'],config['dropout']).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

    train_losses, val_losses = [], []
    best_val_loss    = float('inf')
    patience_counter = 0

    for epoch in range(config['epochs']):
        # Train
        model.train()
        batch_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = np.mean(batch_losses)

        # Validate
        model.eval()
        batch_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                batch_losses.append(criterion(model(X_batch), y_batch).item())
        val_loss = np.mean(batch_losses)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                print(f'  Early stopping at epoch {epoch + 1}')
                break

        if (epoch + 1) % 10 == 0:
            print(f'Epoch: {epoch+1} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')

    model.load_state_dict(torch.load(save_path))
    print(f'Best val loss: {best_val_loss:.4f}')
    return model, train_losses, val_losses

In [9]:
W = CONFIG['window_size']
region_data = {}

all_X_tr, all_y_tr = [], []
all_X_v,  all_y_v  = [], []

global_feature_scaler = MinMaxScaler()
global_target_scaler  = MinMaxScaler()

# Fit global scalers on all non-Virginia data first
non_virginia = df[df['region'] != 'virginia']

for region, group in df.groupby('region'):
    if region == 'virginia':
        print(f'Skipping {region} (held-out test region)')
        continue

    print(f'{region}')
    X_tr, y_tr, X_v, y_v, _, _ = make_splits(
        group, feature_cols, target_col, W, global_feature_scaler, global_target_scaler
    )
    print(f'train={X_tr.shape} | val={X_v.shape}')
    all_X_tr.append(X_tr)
    all_y_tr.append(y_tr)
    all_X_v.append(X_v)
    all_y_v.append(y_v)

virginia = df[df['region'] == 'virginia']
X_test, y_test = make_virginia_test(
    virginia, feature_cols, target_col, W, global_feature_scaler, global_target_scaler
)

X_train = np.concatenate(all_X_tr, axis=0)
y_train = np.concatenate(all_y_tr, axis=0)
X_val = np.concatenate(all_X_v,  axis=0)
y_val = np.concatenate(all_y_v,  axis=0)

print(f'Global train={X_train.shape} | val={X_val.shape} | virginia test={X_test.shape}')

model, train_losses, val_losses = train_model(
    X_train, y_train, X_val, y_val,
    config=CONFIG,
    device=device,
    save_path='best_model_global.pt',
)

california
train=(6964, 24, 16) | val=(1724, 24, 16)
illinois
train=(6964, 24, 16) | val=(1724, 24, 16)
texas
train=(6964, 24, 16) | val=(1724, 24, 16)
Skipping virginia (held-out test region)
Global train=(20892, 24, 16) | val=(5172, 24, 16) | virginia test=(8712, 24, 16)
Epoch: 10 | Train: 0.0021 | Val: 0.0025
Epoch: 20 | Train: 0.0016 | Val: 0.0022
Epoch: 30 | Train: 0.0014 | Val: 0.0017
Epoch: 40 | Train: 0.0013 | Val: 0.0015
Epoch: 50 | Train: 0.0012 | Val: 0.0015
Epoch: 60 | Train: 0.0012 | Val: 0.0014
Epoch: 70 | Train: 0.0011 | Val: 0.0014
Epoch: 80 | Train: 0.0010 | Val: 0.0014
Epoch: 90 | Train: 0.0010 | Val: 0.0014
Epoch: 100 | Train: 0.0010 | Val: 0.0015
  Early stopping at epoch 105
Best val loss: 0.0013


In [10]:
model.eval()

preds_scaled   = []
actuals_scaled = []

with torch.no_grad():
    X_te_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    preds = model(X_te_tensor).cpu().numpy().reshape(-1, 1)  # ensure 2D
    preds_scaled.append(preds)
    actuals_scaled.append(y_test.reshape(-1, 1))

preds_inv = global_target_scaler.inverse_transform(np.concatenate(preds_scaled,   axis=0))
actuals_inv = global_target_scaler.inverse_transform(np.concatenate(actuals_scaled, axis=0))

mae  = mean_absolute_error(actuals_inv, preds_inv)
rmse = np.sqrt(mean_squared_error(actuals_inv, preds_inv))
mape = np.mean(np.abs((actuals_inv - preds_inv) / (actuals_inv + 1e-8))) * 100
r2   = r2_score(actuals_inv, preds_inv)

print(f'Virginia Held-Out Test Results')
print(f'MAE: {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R^2: {r2:.4f}')

Virginia Held-Out Test Results
MAE: 8.4715
RMSE: 11.0928
R^2: 0.8699
